In [2]:
# ==========================================
# 1. التثبيت التلقائي للمكتبات (لحمايتك من الـ Errors)
# ==========================================
import sys
print("⏳ جاري التحقق من المكتبات وتثبيت الناقص منها تلقائياً...")
!{sys.executable} -m pip install pandas numpy scikit-learn openpyxl -q
print("✅ جميع المكتبات جاهزة ومثبتة الآن!")

# ==========================================
# 2. استيراد المكتبات الأساسية
# ==========================================
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import cosine_similarity

# ==========================================
# 3. قراءة ملف مراجعات أمازون
# ==========================================
print("\n⏳ جاري قراءة ملف البيانات الخاص بأمازون...")
try:
    # قراءة الملف (تأكدي أنه بنفس الاسم وفي نفس المجلد)
    df = pd.read_csv('Amazon-Product-Reviews.csv')
    print("✅ تم تحميل البيانات بنجاح من الملف المرفق!")
    print(f"📊 إجمالي عدد الصفوف المكتشفة: {len(df)}")
except Exception as e:
    print(f"⚠️ تنبيه: لم يتم العثور على ملف 'Amazon-Product-Reviews.csv' في هذا المجلد.")
    print("🤖 لتسهيل الأمر وتجربة الكود فوراً، سيقوم النظام بإنشاء بيانات افتراضية ذكية:")
    
    # بناء بيانات وهمية تحاكي أمازون لضمان عمل الكود 100% دون توقف
    mock_data = {
        'ProductId': ['Prod_01', 'Prod_02', 'Prod_03', 'Prod_01', 'Prod_02', 'Prod_03', 'Prod_01', 'Prod_02'],
        'UserId': ['User_A', 'User_A', 'User_B', 'User_C', 'User_C', 'User_D', 'User_E', 'User_E'],
        'Summary': [
            'This product is amazing, high quality!', 
            'FREE CASH CLICK HERE TO WIN MONEY NOW!!!', 
            'Terrible experience, broke on day one.', 
            'Very good value for money, highly recommend.', 
            'SPAM offer buy now win free gifts today',
            'Disappointed with the shipping, but item is okay.',
            'Absolutely fantastic, will buy again.',
            'Worst customer service ever, stay away.'
        ],
        'Score': [5, 1, 1, 4, 1, 3, 5, 1]
    }
    df = pd.DataFrame(mock_data)
    print(f"📊 تم تجهيز {len(df)} نصوص اختبارية مدمجة بنجاح.")

# توحيد أسماء الأعمدة لتفادي الأخطاء البرمجية
if 'Summary' in df.columns:
    df['text'] = df['Summary'].fillna('')
elif 'reviewText' in df.columns:
    df['text'] = df['reviewText'].fillna('')
else:
    df['text'] = "Sample product text"

if 'Score' in df.columns:
    df['rating'] = df['Score']
elif 'overall' in df.columns:
    df['rating'] = df['overall']
else:
    df['rating'] = 5

if 'ProductId' not in df.columns:
    df['ProductId'] = 'Default_Prod'
if 'UserId' not in df.columns:
    df['UserId'] = 'Default_User'

# ==========================================
# 4. دالة تنظيف النصوص (NLP Preprocessing)
# ==========================================
def clean_text(text):
    if not isinstance(text, str): return ""
    text = text.lower()  # تحويل الحروف لصغيرة
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)  # حذف الروابط
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # حذف الأرقام والرموز
    text = ' '.join(text.split())  # إزالة المسافات الزائدة
    return text

df['cleaned_text'] = df['text'].apply(clean_text)

# ==========================================
# 5. نظام تحليل المشاعر (Sentiment: Positive / Negative)
# ==========================================
print("\n🧠 جاري تدريب موديل تحليل المشاعر (NLP)...")
# التقييم 4 و 5 إيجابي (1)، والتقييم 1 و 2 و 3 سلبي (0)
df['sentiment'] = df['rating'].apply(lambda x: 1 if x >= 4 else 0)

vectorizer_sentiment = TfidfVectorizer(max_features=1000)
X_s = vectorizer_sentiment.fit_transform(df['cleaned_text'])
y_s = df['sentiment']

# تدريب الموديل (إذا كانت البيانات قليلة جداً نستخدمها كلها للتدريب لتفادي أخطاء التقسيم)
if len(df) > 5:
    X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_s, y_s, test_size=0.2, random_state=42)
else:
    X_train_s, X_test_s, y_train_s, y_test_s = X_s, X_s, y_s, y_s

model_sentiment = MultinomialNB()
model_sentiment.fit(X_train_s, y_train_s)
print(f"🎯 تم التدريب! دقة موديل المشاعر الحالي: {accuracy_score(y_test_s, model_sentiment.predict(X_test_s))*100:.1f}%")

# ==========================================
# 6. نظام كشف الرسائل المزعجة (Spam / Not Spam)
# ==========================================
print("\n🚫 جاري تدريب موديل كشف الـ Spam والتصيد الإعلاني...")
spam_words = ['buy', 'free', 'win', 'cash', 'money', 'click', 'subscribe', 'offer', 'gifts']
df['is_spam'] = df['cleaned_text'].apply(lambda x: 1 if any(word in x for word in spam_words) else 0)

vectorizer_spam = TfidfVectorizer(max_features=1000)
X_sp = vectorizer_spam.fit_transform(df['cleaned_text'])
y_sp = df['is_spam']

if len(df) > 5:
    X_train_sp, X_test_sp, y_train_sp, y_test_sp = train_test_split(X_sp, y_sp, test_size=0.2, random_state=42)
else:
    X_train_sp, X_test_sp, y_train_sp, y_test_sp = X_sp, X_sp, y_sp, y_sp

model_spam = MultinomialNB()
model_spam.fit(X_train_sp, y_train_sp)
print(f"🎯 تم التدريب! دقة موديل كشف الـ Spam الحالية: {accuracy_score(y_test_sp, model_spam.predict(X_test_sp))*100:.1f}%")

# ==========================================
# 7. نظام الترشيح والتوصية (Recommendation Engine)
# ==========================================
print("\n🛍️ جاري بناء نظام ترشيح المنتجات الذكي البيني...")
try:
    # عمل مصفوفة علاقة المستخدمين بالمنتجات بناء على التقييم
    user_item_matrix = df.pivot_table(index='UserId', columns='ProductId', values='rating').fillna(0)
    # حساب نسب التشابه
    item_similarity = cosine_similarity(user_item_matrix.T)
    item_similarity_df = pd.DataFrame(item_similarity, index=user_item_matrix.columns, columns=user_item_matrix.columns)
    print("✅ تم بناء مصفوفة الترشيحات والتوصيات بنجاح!")
except Exception as e:
    item_similarity_df = None
    print("⚠️ تم تخطي مصفوفة الترشيحات مؤقتاً لتشابه البيانات.")

# ==========================================
# 8. دالة التشغيل الذكية والفحص الشامل
# ==========================================
def smart_ai_assistant(user_review, current_product_id=None):
    print("\n" + "="*60)
    print(f"💬 الفحص الذكي للنص المكتوب: '{user_review}'")
    
    cleaned = clean_text(user_review)
    
    # أ. فحص الـ Spam أولاً حماية للمنصة
    vec_sp = vectorizer_spam.transform([cleaned])
    is_spam_pred = model_spam.predict(vec_sp)[0]
    if is_spam_pred == 1:
        print("🚨 النتيجة: [حظر] هذا النص تم تصنيفه كـ (Spam / ترويج مزعج)!")
        return
    else:
        print("✅ النتيجة: نص طبيعي وسليم (Not Spam).")
    
    # ب. تحليل المشاعر والعاطفة
    vec_s = vectorizer_sentiment.transform([cleaned])
    sentiment_pred = model_sentiment.predict(vec_s)[0]
    sentiment_label = "إيجابي وسعيد (Positive 😊)" if sentiment_pred == 1 else "سلبي أو غاضب (Negative 😡)"
    print(f"📊 تحليل مشاعر العميل: {sentiment_label}")
    
    # ج. تقديم الترشيحات والمنتجات المقترحة للعميل
    if current_product_id and item_similarity_df is not None and current_product_id in item_similarity_df.columns:
        print(f"🎯 وبما أن العميل يتصفح المنتج ({current_product_id})، يقترح النظام له:")
        similar_items = item_similarity_df[current_product_id].sort_values(ascending=False)[1:4]
        for item, score in similar_items.items():
            if score > 0:
                print(f"   - منتج مشابه: {item} (بنسبة توافق وتوصية: {score*100:.1f}%)")
            else:
                print("   - (لا توجد منتجات مشابهة كافية في قاعدة البيانات حالياً)")
                break

# ==========================================
# 9. تشغيل وتجربة مشروعك المكتمل الآن!
# ==========================================
# تجربة 1: نص طبيعي إيجابي لمعرفة رد فعل النظام
smart_ai_assistant("This product is amazing and the quality is very high!", current_product_id=df['ProductId'].iloc[0])

# تجربة 2: نص يحتوي على كلمات سبام دعائية لنرى كيف سيكتشفه ويحظره
smart_ai_assistant("FREE CASH CLICK HERE TO WIN MONEY NOW!!!")

⏳ جاري التحقق من المكتبات وتثبيت الناقص منها تلقائياً...
✅ جميع المكتبات جاهزة ومثبتة الآن!

⏳ جاري قراءة ملف البيانات الخاص بأمازون...
⚠️ تنبيه: لم يتم العثور على ملف 'Amazon-Product-Reviews.csv' في هذا المجلد.
🤖 لتسهيل الأمر وتجربة الكود فوراً، سيقوم النظام بإنشاء بيانات افتراضية ذكية:
📊 تم تجهيز 8 نصوص اختبارية مدمجة بنجاح.

🧠 جاري تدريب موديل تحليل المشاعر (NLP)...
🎯 تم التدريب! دقة موديل المشاعر الحالي: 50.0%

🚫 جاري تدريب موديل كشف الـ Spam والتصيد الإعلاني...
🎯 تم التدريب! دقة موديل كشف الـ Spam الحالية: 100.0%

🛍️ جاري بناء نظام ترشيح المنتجات الذكي البيني...
✅ تم بناء مصفوفة الترشيحات والتوصيات بنجاح!

💬 الفحص الذكي للنص المكتوب: 'This product is amazing and the quality is very high!'
✅ النتيجة: نص طبيعي وسليم (Not Spam).
📊 تحليل مشاعر العميل: إيجابي وسعيد (Positive 😊)
🎯 وبما أن العميل يتصفح المنتج (Prod_01)، يقترح النظام له:
   - منتج مشابه: Prod_02 (بنسبة توافق وتوصية: 99.5%)
   - (لا توجد منتجات مشابهة كافية في قاعدة البيانات حالياً)

💬 الفحص الذكي للنص المكتوب: 'FREE CASH 

In [3]:
# ==========================================
# 1. التثبيت التلقائي للمكتبات (لحمايتك من الـ Errors)
# ==========================================
import sys
print("⏳ جاري التحقق من المكتبات وتثبيت الناقص منها تلقائياً...")
!{sys.executable} -m pip install pandas numpy scikit-learn openpyxl -q
print("✅ جميع المكتبات جاهزة ومثبتة الآن!")

# ==========================================
# 2. استيراد المكتبات الأساسية
# ==========================================
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import cosine_similarity

# ==========================================
# 3. قراءة ملف مراجعات أمازون
# ==========================================
print("\n⏳ جاري قراءة ملف البيانات الخاص بأمازون...")
try:
    # قراءة الملف (تأكدي أنه بنفس الاسم وفي نفس المجلد)
    df = pd.read_csv('Amazon-Product-Reviews.csv')
    print("✅ تم تحميل البيانات بنجاح من الملف المرفق!")
    print(f"📊 إجمالي عدد الصفوف المكتشفة: {len(df)}")
except Exception as e:
    print(f"⚠️ تنبيه: لم يتم العثور على ملف 'Amazon-Product-Reviews.csv' في هذا المجلد.")
    print("🤖 لتسهيل الأمر وتجربة الكود فوراً، سيقوم النظام بإنشاء بيانات افتراضية ذكية:")
    
    # بناء بيانات وهمية تحاكي أمازون لضمان عمل الكود 100% دون توقف
    mock_data = {
        'ProductId': ['Prod_01', 'Prod_02', 'Prod_03', 'Prod_01', 'Prod_02', 'Prod_03', 'Prod_01', 'Prod_02'],
        'UserId': ['User_A', 'User_A', 'User_B', 'User_C', 'User_C', 'User_D', 'User_E', 'User_E'],
        'Summary': [
            'This product is amazing, high quality!', 
            'FREE CASH CLICK HERE TO WIN MONEY NOW!!!', 
            'Terrible experience, broke on day one.', 
            'Very good value for money, highly recommend.', 
            'SPAM offer buy now win free gifts today',
            'Disappointed with the shipping, but item is okay.',
            'Absolutely fantastic, will buy again.',
            'Worst customer service ever, stay away.'
        ],
        'Score': [5, 1, 1, 4, 1, 3, 5, 1]
    }
    df = pd.DataFrame(mock_data)
    print(f"📊 تم تجهيز {len(df)} نصوص اختبارية مدمجة بنجاح.")

# توحيد أسماء الأعمدة لتفادي الأخطاء البرمجية
if 'Summary' in df.columns:
    df['text'] = df['Summary'].fillna('')
elif 'reviewText' in df.columns:
    df['text'] = df['reviewText'].fillna('')
else:
    df['text'] = "Sample product text"

if 'Score' in df.columns:
    df['rating'] = df['Score']
elif 'overall' in df.columns:
    df['rating'] = df['overall']
else:
    df['rating'] = 5

if 'ProductId' not in df.columns:
    df['ProductId'] = 'Default_Prod'
if 'UserId' not in df.columns:
    df['UserId'] = 'Default_User'

# ==========================================
# 4. دالة تنظيف النصوص (NLP Preprocessing)
# ==========================================
def clean_text(text):
    if not isinstance(text, str): return ""
    text = text.lower()  # تحويل الحروف لصغيرة
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)  # حذف الروابط
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # حذف الأرقام والرموز
    text = ' '.join(text.split())  # إزالة المسافات الزائدة
    return text

df['cleaned_text'] = df['text'].apply(clean_text)

# ==========================================
# 5. نظام تحليل المشاعر (Sentiment: Positive / Negative)
# ==========================================
print("\n🧠 جاري تدريب موديل تحليل المشاعر (NLP)...")
# التقييم 4 و 5 إيجابي (1)، والتقييم 1 و 2 و 3 سلبي (0)
df['sentiment'] = df['rating'].apply(lambda x: 1 if x >= 4 else 0)

vectorizer_sentiment = TfidfVectorizer(max_features=1000)
X_s = vectorizer_sentiment.fit_transform(df['cleaned_text'])
y_s = df['sentiment']

# تدريب الموديل (إذا كانت البيانات قليلة جداً نستخدمها كلها للتدريب لتفادي أخطاء التقسيم)
if len(df) > 5:
    X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_s, y_s, test_size=0.2, random_state=42)
else:
    X_train_s, X_test_s, y_train_s, y_test_s = X_s, X_s, y_s, y_s

model_sentiment = MultinomialNB()
model_sentiment.fit(X_train_s, y_train_s)
print(f"🎯 تم التدريب! دقة موديل المشاعر الحالي: {accuracy_score(y_test_s, model_sentiment.predict(X_test_s))*100:.1f}%")

# ==========================================
# 6. نظام كشف الرسائل المزعجة (Spam / Not Spam)
# ==========================================
print("\n🚫 جاري تدريب موديل كشف الـ Spam والتصيد الإعلاني...")
spam_words = ['buy', 'free', 'win', 'cash', 'money', 'click', 'subscribe', 'offer', 'gifts']
df['is_spam'] = df['cleaned_text'].apply(lambda x: 1 if any(word in x for word in spam_words) else 0)

vectorizer_spam = TfidfVectorizer(max_features=1000)
X_sp = vectorizer_spam.fit_transform(df['cleaned_text'])
y_sp = df['is_spam']

if len(df) > 5:
    X_train_sp, X_test_sp, y_train_sp, y_test_sp = train_test_split(X_sp, y_sp, test_size=0.2, random_state=42)
else:
    X_train_sp, X_test_sp, y_train_sp, y_test_sp = X_sp, X_sp, y_sp, y_sp

model_spam = MultinomialNB()
model_spam.fit(X_train_sp, y_train_sp)
print(f"🎯 تم التدريب! دقة موديل كشف الـ Spam الحالية: {accuracy_score(y_test_sp, model_spam.predict(X_test_sp))*100:.1f}%")

# ==========================================
# 7. نظام الترشيح والتوصية (Recommendation Engine)
# ==========================================
print("\n🛍️ جاري بناء نظام ترشيح المنتجات الذكي البيني...")
try:
    # عمل مصفوفة علاقة المستخدمين بالمنتجات بناء على التقييم
    user_item_matrix = df.pivot_table(index='UserId', columns='ProductId', values='rating').fillna(0)
    # حساب نسب التشابه
    item_similarity = cosine_similarity(user_item_matrix.T)
    item_similarity_df = pd.DataFrame(item_similarity, index=user_item_matrix.columns, columns=user_item_matrix.columns)
    print("✅ تم بناء مصفوفة الترشيحات والتوصيات بنجاح!")
except Exception as e:
    item_similarity_df = None
    print("⚠️ تم تخطي مصفوفة الترشيحات مؤقتاً لتشابه البيانات.")

# ==========================================
# 8. دالة التشغيل الذكية والفحص الشامل
# ==========================================
def smart_ai_assistant(user_review, current_product_id=None):
    print("\n" + "="*60)
    print(f"💬 الفحص الذكي للنص المكتوب: '{user_review}'")
    
    cleaned = clean_text(user_review)
    
    # أ. فحص الـ Spam أولاً حماية للمنصة
    vec_sp = vectorizer_spam.transform([cleaned])
    is_spam_pred = model_spam.predict(vec_sp)[0]
    if is_spam_pred == 1:
        print("🚨 النتيجة: [حظر] هذا النص تم تصنيفه كـ (Spam / ترويج مزعج)!")
        return
    else:
        print("✅ النتيجة: نص طبيعي وسليم (Not Spam).")
    
    # ب. تحليل المشاعر والعاطفة
    vec_s = vectorizer_sentiment.transform([cleaned])
    sentiment_pred = model_sentiment.predict(vec_s)[0]
    sentiment_label = "إيجابي وسعيد (Positive 😊)" if sentiment_pred == 1 else "سلبي أو غاضب (Negative 😡)"
    print(f"📊 تحليل مشاعر العميل: {sentiment_label}")
    
    # ج. تقديم الترشيحات والمنتجات المقترحة للعميل
    if current_product_id and item_similarity_df is not None and current_product_id in item_similarity_df.columns:
        print(f"🎯 وبما أن العميل يتصفح المنتج ({current_product_id})، يقترح النظام له:")
        similar_items = item_similarity_df[current_product_id].sort_values(ascending=False)[1:4]
        for item, score in similar_items.items():
            if score > 0:
                print(f"   - منتج مشابه: {item} (بنسبة توافق وتوصية: {score*100:.1f}%)")
            else:
                print("   - (لا توجد منتجات مشابهة كافية في قاعدة البيانات حالياً)")
                break

# ==========================================
# 9. تشغيل وتجربة مشروعك المكتمل الآن!
# ==========================================
# تجربة 1: نص طبيعي إيجابي لمعرفة رد فعل النظام
smart_ai_assistant("This product is amazing and the quality is very high!", current_product_id=df['ProductId'].iloc[0])

# تجربة 2: نص يحتوي على كلمات سبام دعائية لنرى كيف سيكتشفه ويحظره
smart_ai_assistant("FREE CASH CLICK HERE TO WIN MONEY NOW!!!")

⏳ جاري التحقق من المكتبات وتثبيت الناقص منها تلقائياً...
✅ جميع المكتبات جاهزة ومثبتة الآن!

⏳ جاري قراءة ملف البيانات الخاص بأمازون...
⚠️ تنبيه: لم يتم العثور على ملف 'Amazon-Product-Reviews.csv' في هذا المجلد.
🤖 لتسهيل الأمر وتجربة الكود فوراً، سيقوم النظام بإنشاء بيانات افتراضية ذكية:
📊 تم تجهيز 8 نصوص اختبارية مدمجة بنجاح.

🧠 جاري تدريب موديل تحليل المشاعر (NLP)...
🎯 تم التدريب! دقة موديل المشاعر الحالي: 50.0%

🚫 جاري تدريب موديل كشف الـ Spam والتصيد الإعلاني...
🎯 تم التدريب! دقة موديل كشف الـ Spam الحالية: 100.0%

🛍️ جاري بناء نظام ترشيح المنتجات الذكي البيني...
✅ تم بناء مصفوفة الترشيحات والتوصيات بنجاح!

💬 الفحص الذكي للنص المكتوب: 'This product is amazing and the quality is very high!'
✅ النتيجة: نص طبيعي وسليم (Not Spam).
📊 تحليل مشاعر العميل: إيجابي وسعيد (Positive 😊)
🎯 وبما أن العميل يتصفح المنتج (Prod_01)، يقترح النظام له:
   - منتج مشابه: Prod_02 (بنسبة توافق وتوصية: 99.5%)
   - (لا توجد منتجات مشابهة كافية في قاعدة البيانات حالياً)

💬 الفحص الذكي للنص المكتوب: 'FREE CASH 